# Serving benchmark: one 8B model, one GPU, five questions

Qwen3-8B served with **vLLM 0.32.0** and measured with `vllm bench serve`. Everything runs from a script in the
background and saves after every run, so a disconnect costs nothing: re-run the cells and it carries on.

| # | Question | Experiment | Lesson idea |
|---|---|---|---|
| 1 | How many users can one GPU serve, and how fast does each one feel it? | concurrency sweep 1 → 128 | batching trades per-user speed for throughput |
| 2 | Does a smaller number format pay off? | the same sweep in BF16, FP8, INT4 (AWQ) | fewer bytes per weight → faster decode, more KV cache |
| 3 | What do long agent prompts cost? | 8k-token prompts vs 1k | prefill dominates time to first token |
| 4 | What does a shared system prompt save? | prefix caching on vs off | same prefix → same K/V, computed once |
| 5 | When does speculative decoding help? | EAGLE-3 and n-gram, code vs chat, low vs high concurrency | spare compute at low batch; gone at high batch |

**GPU:** anything with compute capability ≥ 8.0 (L4, A100, L40S, H100). Not a T4. The plan sizes itself to the
GPU's memory. **Time, my estimate:** H100 ~1–1.5 h, A100 ~1.5–2 h, L4 ~3–4 h, of which ~10 min is setup.

**Before you start the GPU clock:** read sections 1–5 now; they're the same on any machine. Then run the cells top
to bottom. The only waits are the install (~3–5 min), the preflight (~5 min) and the main run.

## 1. Machine check

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

COLAB = "google.colab" in sys.modules
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu or "No GPU visible. On Colab: Runtime → Change runtime type → L4 or A100.")
name, mem, cc, drv = [x.strip() for x in gpu.split(",")]
assert float(cc) >= 8.0, f"{name} (compute {cc}) lacks BF16 and modern kernels; pick an L4, A100, L40S or H100."
print("FP8 path:", "native W8A8 (Ada/Hopper tensor cores)" if float(cc) >= 8.9 else
      "weight-only (Marlin W8A16): Ampere has no FP8 tensor cores, so FP8 saves memory but not compute")

In [ ]:
# Where results go. On Colab, Google Drive, so they survive the runtime being recycled.
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/llm-systems-lab/results-" + name.replace(" ", "_"))
    WORK = Path("/content/lab")
else:
    OUT = Path.home() / ("results-" + name.replace(" ", "_"))
    WORK = Path.home() / "lab"
OUT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True); os.chdir(WORK)
print("results →", OUT)

## 2. Install vLLM and start the model downloads

vLLM runs as a separate process (the CLI), so it never touches this notebook's Python. The four repos (~32 GB) download
in the background while the install finishes. A Hugging Face token is optional (none of these models are gated); on
Colab, add one as the secret `HF_TOKEN` for faster, rate-limit-free downloads.

In [ ]:
# Installs into this kernel's own Python, whatever the image, and finds the CLIs next to it.
import shutil
PY = sys.executable
!{PY} -m pip install -q uv
!{PY} -m uv pip install --python {PY} -q "vllm[bench]==0.32.0" pandas matplotlib
BIN = Path(PY).parent
VLLM = shutil.which("vllm") or str(BIN / "vllm")
HF = shutil.which("hf") or str(BIN / "hf")
os.environ["VLLM_BIN"] = VLLM
!{VLLM} --version

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
models = ["Qwen/Qwen3-8B", "Qwen/Qwen3-8B-FP8", "Qwen/Qwen3-8B-AWQ", "RedHatAI/Qwen3-8B-speculator.eagle3"]
dl = [subprocess.Popen([HF, "download", m], stdout=open(f"download-{m.split('/')[1]}.log", "w"),
                       stderr=subprocess.STDOUT) for m in models]
print("downloading in the background:", *models, sep="\n  ")

In [ ]:
# Re-run to check. Each line says done or still going.
for m, p in zip(models, dl):
    print(f"{m:45s}", "done" if p.poll() == 0 else "still going" if p.poll() is None else f"FAILED, see download-{m.split('/')[1]}.log")
!df -h . | tail -1

## 3. The code

`runner.py` starts one vLLM server per configuration, runs `vllm bench serve` against it at each concurrency level and
saves every run to `OUT/runs/`. Before and after each run it reads the server's `/metrics` counters, which is how it gets
the prefix-cache hit rate and the speculative-decoding acceptance length. `analyze.py` turns the folder into a CSV,
charts, a cost table and a findings draft. Both are tested on a laptop against a fake `vllm` (see `tests/`).

In [ ]:
%%writefile runner.py
"""Serving benchmark runner: starts one vLLM server at a time and runs `vllm bench serve` against it.

Resume-safe: a run whose result file already exists is skipped, so after a disconnect you re-run the same command and
it carries on. Failed runs leave an `.error` file and are retried on the next start.

    python runner.py --out results            # the full plan for this GPU
    python runner.py --out results --preflight  # one tiny run (~3 min) to check the setup end to end
    python runner.py --out results --plan     # print the plan and exit
"""
import argparse, json, os, re, signal, subprocess, sys, time, urllib.request
from pathlib import Path

VLLM = os.environ.get("VLLM_BIN", "vllm")
PORT = int(os.environ.get("VLLM_PORT", 8000))
BASE = f"http://127.0.0.1:{PORT}"

BF16, FP8, INT4 = "Qwen/Qwen3-8B", "Qwen/Qwen3-8B-FP8", "Qwen/Qwen3-8B-AWQ"
DRAFT = "RedHatAI/Qwen3-8B-speculator.eagle3"
MODELS = [BF16, FP8, INT4, DRAFT]

# One server per configuration. All serve the same 8B model, so only the named setting differs.
SERVERS = {
    "bf16": (BF16, []),
    "fp8": (FP8, []),
    "int4": (INT4, []),
    "bf16-nocache": (BF16, ["--no-enable-prefix-caching"]),
    "bf16-eagle3": (BF16, ["--speculative-config", json.dumps(
        {"method": "eagle3", "model": DRAFT, "num_speculative_tokens": 3})]),
    "bf16-ngram": (BF16, ["--speculative-config", json.dumps(
        {"method": "ngram", "num_speculative_tokens": 4, "prompt_lookup_min": 2, "prompt_lookup_max": 4})]),
}
# Context length and memory share per profile. On a 24 GB L4 the bf16 weights take ~15.3 GiB, which leaves room for
# roughly 25-30k tokens of KV cache, so 16k contexts may not fit; 12k still holds an 8k agent prompt.
SERVER_ARGS = {"large": ["--max-model-len", "16384", "--gpu-memory-utilization", "0.90"],
               "medium": ["--max-model-len", "16384", "--gpu-memory-utilization", "0.90"],
               "small": ["--max-model-len", "12288", "--gpu-memory-utilization", "0.92"]}

# Workloads. Random prompts with --ignore-eos give fixed lengths, so runs compare like for like.
WORKLOADS = {
    "chat": ["--dataset-name", "random", "--random-input-len", "1024", "--random-output-len", "256", "--ignore-eos"],
    # Shaped like an agent turn: a long prompt (system prompt, tool docs, results), a short answer.
    "agent": ["--dataset-name", "random", "--random-input-len", "8192", "--random-output-len", "256", "--ignore-eos"],
    # 4 shared 6,000-token prefixes (think: 4 agents' system prompts), each followed by a 500-token question.
    "prefix": ["--dataset-name", "prefix_repetition", "--prefix-repetition-prefix-len", "6000",
               "--prefix-repetition-suffix-len", "500", "--prefix-repetition-num-prefixes", "4",
               "--prefix-repetition-output-len", "128", "--ignore-eos"],
    # Real text, greedy, so the draft's guesses can actually be right. Thinking off to keep answers comparable.
    "spec-code": ["--dataset-name", "custom", "--dataset-path", "{prompts}/code_edit.jsonl",
                  "--custom-output-len", "256", "--temperature", "0",
                  "--chat-template-kwargs", '{"enable_thinking": false}'],
    "spec-chat": ["--dataset-name", "custom", "--dataset-path", "{prompts}/open_chat.jsonl",
                  "--custom-output-len", "256", "--temperature", "0",
                  "--chat-template-kwargs", '{"enable_thinking": false}'],
}

# Concurrency levels by GPU memory. A 24 GB L4 holds the 16 GB bf16 weights plus only ~4 GB of KV cache.
PROFILES = {
    "large": dict(chat=[1, 4, 16, 64, 128], agent=[1, 4, 16, 32], prefix=[8, 32], spec=[1, 16, 64]),  # 80 GB
    "medium": dict(chat=[1, 4, 16, 64], agent=[1, 4, 8, 16], prefix=[8, 16], spec=[1, 8, 32]),  # 40-48 GB
    "small": dict(chat=[1, 4, 16, 32], agent=[1, 2, 4, 8], prefix=[4, 8], spec=[1, 8, 16]),  # 24 GB
}
SEEDS = {w: 1000 * i for i, w in enumerate(WORKLOADS)}

# Server metrics read before and after each run (Prometheus counters gain a _total suffix).
COUNTERS = ["vllm:prefix_cache_queries", "vllm:prefix_cache_hits", "vllm:num_preemptions",
            "vllm:spec_decode_num_drafts", "vllm:spec_decode_num_draft_tokens", "vllm:spec_decode_num_accepted_tokens",
            "vllm:prompt_tokens", "vllm:generation_tokens"]


def gpu_info():
    if "GPU_INFO" in os.environ:  # tests on a machine without a GPU
        name, mem, cc = os.environ["GPU_INFO"].split(";")
    else:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, check=True).stdout.splitlines()[0]
        name, mem, cc = [x.strip() for x in out.split(",")]
    mem_gb = float(mem) / 1024
    return {"name": name, "memory_gb": round(mem_gb, 1), "compute_cap": cc,
            "profile": "large" if mem_gb >= 70 else "medium" if mem_gb >= 38 else "small"}


def plan(profile):
    """Every run, grouped by server so each server starts once."""
    p, runs = PROFILES[profile], []
    def add(server, workload, conc, n):
        runs.append({"id": f"{server}__{workload}__c{conc}", "server": server, "workload": workload,
                     "concurrency": conc, "num_prompts": n, "seed": SEEDS[workload] + conc})
    for s in ["bf16", "fp8", "int4"]:
        for c in p["chat"]: add(s, "chat", c, max(20, 4 * c))
        for c in p["agent"]: add(s, "agent", c, max(20, 4 * c))
    for s in ["bf16", "bf16-nocache"]:
        for c in p["prefix"]: add(s, "prefix", c, 128)
    for s in ["bf16", "bf16-eagle3", "bf16-ngram"]:
        for w in ["spec-code", "spec-chat"]:
            for c in p["spec"]: add(s, w, c, max(24, 3 * c))
    order = list(SERVERS)
    return sorted(runs, key=lambda r: order.index(r["server"]))


def log(out, msg):
    line = f"{time.strftime('%H:%M:%S')}  {msg}"
    print(line, flush=True)
    with open(out / "progress.log", "a") as f: f.write(line + "\n")


def get(path, timeout=5):
    with urllib.request.urlopen(BASE + path, timeout=timeout) as r: return r.status, r.read().decode()


def scrape():
    """Sum each counter over its label sets."""
    _, text = get("/metrics")
    vals = dict.fromkeys(COUNTERS, 0.0)
    for line in text.splitlines():
        if line.startswith("#"): continue
        m = re.match(r"([a-zA-Z_:]+?)(_total)?(\{.*\})?\s+([0-9.eE+-]+)$", line)
        if m and m.group(1) in vals: vals[m.group(1)] += float(m.group(4))
    return vals


def start_server(name, out, profile):
    model, extra = SERVERS[name]
    import socket
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", PORT)) == 0:
            raise RuntimeError(f"port {PORT} is already in use (an old server?); stop it or set VLLM_PORT")
    logf = out / "logs" / f"server-{name}.log"
    args = list(SERVER_ARGS[profile])
    if name in ("bf16-eagle3", "bf16-ngram"):
        # Spec-decoding prompts are under 2k tokens. A short context leaves room for the 2 GB EAGLE-3 draft on a 24 GB L4.
        args[args.index("--max-model-len") + 1] = "4096"
    cmd = [VLLM, "serve", model, *args, "--port", str(PORT), *extra]
    proc = subprocess.Popen(cmd, stdout=open(logf, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < 1500:
        if proc.poll() is not None:
            raise RuntimeError(f"server {name} exited; last log lines:\n" + "".join(logf.read_text().splitlines(True)[-25:]))
        try:
            if get("/health", 2)[0] == 200: break
        except Exception: time.sleep(5)
    else:
        stop_server(proc); raise RuntimeError(f"server {name} not healthy after 25 min")
    text = logf.read_text()
    kv = re.search(r"KV cache size: ([\d,]+) tokens, Maximum concurrency for ([\d,]+) tokens per request: ([\d.]+)x", text)
    load = re.search(r"Model loading took ([\d.]+) GiB memory and ([\d.]+) seconds", text)
    info = {"server": name, "model": model, "args": extra, "startup_s": round(time.time() - t0, 1),
            "kv_cache_tokens": int(kv.group(1).replace(",", "")) if kv else None,
            "kv_max_concurrency_at_max_len": float(kv.group(3)) if kv else None,
            "weights_gib": float(load.group(1)) if load else None}
    return proc, info


def stop_server(proc):
    try: os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError: return
    try: proc.wait(90)
    except subprocess.TimeoutExpired: os.killpg(proc.pid, signal.SIGKILL); proc.wait()
    time.sleep(5)  # let the GPU memory go before the next server starts


def bench(run, out, gpu):
    model = SERVERS[run["server"]][0]
    wl = [a.replace("{prompts}", str(out / "prompts")) for a in WORKLOADS[run["workload"]]]
    tmp = out / "raw"
    cmd = [VLLM, "bench", "serve", "--backend", "openai", "--base-url", BASE, "--model", model,
           "--num-prompts", str(run["num_prompts"]), "--max-concurrency", str(run["concurrency"]),
           "--seed", str(run["seed"]), "--num-warmups", "2",
           "--percentile-metrics", "ttft,tpot,itl,e2el", "--metric-percentiles", "50,90,95,99",
           "--save-result", "--result-dir", str(tmp), "--result-filename", run["id"] + ".json", *wl]
    before, t0 = scrape(), time.time()
    with open(out / "logs" / f"bench-{run['id']}.log", "w") as f:
        subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, check=True, timeout=1800)
    after = scrape()
    res = json.loads((tmp / (run["id"] + ".json")).read_text())
    res.update(run=run, gpu=gpu, wall_s=round(time.time() - t0, 1),
               server_counters={k: after[k] - before[k] for k in COUNTERS})
    (out / "runs" / (run["id"] + ".json")).write_text(json.dumps(res, indent=1))


def write_prompts(d):
    """Two prompt sets for speculative decoding: code rewrites (predictable output) and open questions."""
    import inspect, importlib
    d.mkdir(parents=True, exist_ok=True)
    funcs = []
    for mod in ["textwrap", "difflib", "statistics", "calendar", "fractions", "shlex", "csv", "argparse", "heapq",
                "ipaddress", "configparser", "string", "base64", "json.decoder", "json.encoder", "pprint"]:
        m = importlib.import_module(mod)
        for _, obj in sorted(vars(m).items()):
            if inspect.isfunction(obj) and obj.__module__ == m.__name__:
                try: src = inspect.getsource(obj)
                except OSError: continue
                if 15 <= src.count("\n") <= 60: funcs.append(src)
    funcs = funcs[:48]
    with open(d / "code_edit.jsonl", "w") as f:
        for src in funcs:
            f.write(json.dumps({"prompt": "Add Python type hints to this function and rename any one-letter "
                                "variables to descriptive names. Return only the full updated code.\n\n"
                                f"```python\n{src}```"}) + "\n")
    qs = ["Explain how a refrigerator keeps food cold", "Write a short story about a lighthouse keeper who finds a map",
          "What are the pros and cons of remote work for a small team", "Describe the water cycle to a ten-year-old",
          "Plan a three-day trip to Kyoto for someone who likes food and temples", "Why did the Roman Empire split in two",
          "Give advice to someone preparing for their first marathon", "Compare tea and coffee culture around the world",
          "Explain compound interest with an example", "Write a product description for a solar-powered backpack",
          "What makes a good team meeting", "Describe how vaccines train the immune system",
          "Write a letter from a cat to its owner about the new dog", "How do airplanes stay in the air",
          "Summarise the causes and effects of the 2008 financial crisis", "Suggest a weekly plan to learn guitar",
          "Explain what a black hole is and how we detect them", "Write a poem about monsoon rain in Mumbai",
          "What should a first-time manager focus on in their first 90 days", "How does a bill become law in India",
          "Describe the life cycle of a star", "Write a dialogue between a chef and a food critic",
          "Explain the difference between weather and climate", "How should a beginner start investing",
          "Describe a futuristic city in the year 2200", "What are the health benefits of regular sleep",
          "Explain how the internet routes a message across the world", "Write a speech for a school sports day",
          "Why do languages change over time", "Give tips for writing a clear email to a busy executive",
          "Describe how chess openings are learned and why they matter", "Explain photosynthesis step by step"]
    with open(d / "open_chat.jsonl", "w") as f:
        for q in qs: f.write(json.dumps({"prompt": q + ". Answer in about 300 words."}) + "\n")
    return len(funcs), len(qs)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--preflight", action="store_true")
    ap.add_argument("--plan", action="store_true")
    ap.add_argument("--profile", choices=list(PROFILES))
    ap.add_argument("--servers", help="comma-separated subset, e.g. bf16,fp8")
    a = ap.parse_args()

    gpu = gpu_info()
    profile = a.profile or gpu["profile"]
    out = a.out / "preflight" if a.preflight else a.out
    for sub in ["runs", "raw", "logs"]: (out / sub).mkdir(parents=True, exist_ok=True)
    runs = ([{"id": "bf16__chat__c4", "server": "bf16", "workload": "chat", "concurrency": 4, "num_prompts": 8,
              "seed": 4}] if a.preflight else plan(profile))
    if a.servers: runs = [r for r in runs if r["server"] in a.servers.split(",")]
    if a.plan:
        for r in runs: print(r["id"], r["num_prompts"])
        print(f"{len(runs)} runs on {gpu['name']} ({profile})"); return

    write_prompts(out / "prompts")
    ver = subprocess.run([VLLM, "--version"], capture_output=True, text=True).stdout.strip()
    (out / "env.json").write_text(json.dumps({**gpu, "profile": profile, "vllm": ver}, indent=1))
    servers_file = out / "servers.json"
    servers = json.loads(servers_file.read_text()) if servers_file.exists() else {}

    todo = [r for r in runs if not (out / "runs" / (r["id"] + ".json")).exists()]
    log(out, f"{gpu['name']} · profile {profile} · vLLM {ver} · {len(runs) - len(todo)} done, {len(todo)} to run")
    failed = 0
    for name in dict.fromkeys(r["server"] for r in todo):
        group = [r for r in todo if r["server"] == name]
        log(out, f"starting server {name} ({len(group)} runs)")
        try:
            proc, info = start_server(name, out, profile)
        except RuntimeError as e:
            log(out, f"FAILED to start {name}: {e}"); failed += len(group); continue
        servers[name] = info; servers_file.write_text(json.dumps(servers, indent=1))
        log(out, f"  up in {info['startup_s']} s · KV cache {info['kv_cache_tokens']} tokens · weights {info['weights_gib']} GiB")
        try:
            for r in group:
                err = out / "runs" / (r["id"] + ".error")
                try:
                    bench(r, out, gpu); err.unlink(missing_ok=True)
                    res = json.loads((out / "runs" / (r["id"] + ".json")).read_text())
                    log(out, f"  {r['id']}: {res['output_throughput']:.0f} out tok/s · p50 TTFT {res['median_ttft_ms']:.0f} ms"
                             f" · p50 TPOT {res['median_tpot_ms']:.1f} ms · {res['completed']}/{r['num_prompts']} ok")
                except Exception as e:
                    failed += 1; err.write_text(repr(e)); log(out, f"  {r['id']}: FAILED ({e!r:.200}), see logs/")
        finally:
            stop_server(proc)
    log(out, f"finished · {failed} failed" + (" · re-run the same command to retry them" if failed else ""))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze.py
"""Turns a results folder from runner.py into a table, four charts, a cost table and a findings draft.

    python analyze.py results            # writes results/results.csv, results/figures/*.png, results/findings.md
    python analyze.py results --price 2.69   # $/GPU-hour if the GPU isn't in PRICES
"""
import argparse, json, math
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# $/GPU-hour, RunPod Community Cloud list prices on 2026-10-25 (runpod.io/pricing). Colab bills in compute units, so
# cost is always computed at what renting the same GPU would cost.
PRICES = [("H100 NVL", 2.59), ("H100 PCIe", 1.99), ("H100", 2.69), ("H200", 3.59), ("A100-SXM4-80GB", 1.39),
          ("A100 80GB PCIe", 1.19), ("L40S", 0.79), ("L4", 0.44)]
SLO = {"chat": {"ttft": 1000, "tpot": 50}, "agent": {"ttft": 3000, "tpot": 50}}  # p95, ms. 50 ms/token = 20 tok/s

BLUE, ORANGE, GREEN, GRAY, INK2, GRID = "#2a78d6", "#eb6834", "#2e9e6a", "#8a8984", "#52514e", "#e4e3df"
COLOR = {"bf16": BLUE, "fp8": ORANGE, "int4": GREEN, "bf16-eagle3": ORANGE, "bf16-ngram": GREEN, "bf16-nocache": GRAY}
LABEL = {"bf16": "BF16", "fp8": "FP8", "int4": "INT4 (AWQ)", "bf16-eagle3": "EAGLE-3 draft", "bf16-ngram": "n-gram lookup",
         "bf16-nocache": "prefix cache off"}
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.color": GRID, "axes.edgecolor": GRID,
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2, "lines.markersize": 7})


def load(d):
    rows = []
    for f in sorted((d / "runs").glob("*.json")):
        r = json.loads(f.read_text()); run, c = r["run"], r.get("server_counters", {})
        drafts = c.get("vllm:spec_decode_num_drafts", 0)
        q = c.get("vllm:prefix_cache_queries", 0)
        rows.append({**{k: run[k] for k in ["id", "server", "workload", "concurrency", "num_prompts"]},
                     "completed": r["completed"], "failed": r.get("failed", 0),
                     "out_tok_s": r["output_throughput"], "total_tok_s": r["total_token_throughput"],
                     "req_s": r["request_throughput"],
                     **{f"{p}_{m}": r.get(f"{'median' if p == 'p50' else p}_{m}_ms") for m in ["ttft", "tpot", "itl", "e2el"]
                        for p in ["p50", "p95", "p99"]},
                     "user_tok_s": 1000 / r["median_tpot_ms"] if r.get("median_tpot_ms") else None,
                     "prefix_hit_rate": c.get("vllm:prefix_cache_hits", 0) / q if q else None,
                     "accept_len": 1 + c.get("vllm:spec_decode_num_accepted_tokens", 0) / drafts if drafts else None,
                     "preemptions": c.get("vllm:num_preemptions", 0)})
    env = json.loads((d / "env.json").read_text())
    servers = json.loads((d / "servers.json").read_text()) if (d / "servers.json").exists() else {}
    return pd.DataFrame(rows), env, servers


def price_for(gpu, override):
    if override: return override
    return next((p for k, p in PRICES if k in gpu), None)


def fig_frontier(df, out, gpu):
    """Throughput per GPU vs speed each user sees: the serving trade-off in one picture."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, wl in zip(axes, ["chat", "agent"]):
        for s in ["bf16", "fp8", "int4"]:
            d = df[(df.server == s) & (df.workload == wl)].sort_values("concurrency")
            if d.empty: continue
            ax.plot(d.user_tok_s, d.out_tok_s, "o-", color=COLOR[s], label=LABEL[s])
            for _, r in d.iterrows():
                ax.annotate(f"{r.concurrency}", (r.user_tok_s, r.out_tok_s), textcoords="offset points",
                            xytext=(5, 4), fontsize=8, color=INK2)
        ax.axvline(1000 / SLO[wl]["tpot"], color=GRAY, ls="--", lw=1)
        ax.set(title=f"{wl}: {'1k in / 256 out' if wl == 'chat' else '8k in / 256 out'}",
               xlabel="tokens/s each user sees (1000 / p50 TPOT)", ylabel="output tokens/s per GPU")
    axes[0].legend(frameon=False)
    fig.suptitle(f"Throughput vs per-user speed, Qwen3-8B on {gpu} (labels = concurrent requests; dashed = 20 tok/s)",
                 fontsize=10, color=INK2)
    fig.tight_layout(); fig.savefig(out / "1_frontier.png"); plt.close(fig)


def fig_ttft(df, out):
    fig, ax = plt.subplots(figsize=(6, 4))
    for s in ["bf16", "fp8", "int4"]:
        d = df[(df.server == s) & (df.workload == "agent")].sort_values("concurrency")
        if not d.empty: ax.plot(d.concurrency, d.p50_ttft, "o-", color=COLOR[s], label=LABEL[s])
    ax.set_xscale("log", base=2)
    ax.set(title="Agent-shaped prompts (8k tokens): time to first token", xlabel="concurrent requests",
           ylabel="p50 TTFT (ms)")
    ax.legend(frameon=False); fig.tight_layout(); fig.savefig(out / "2_ttft_agent.png"); plt.close(fig)


def fig_prefix(df, out):
    d = df[df.workload == "prefix"]
    if d.empty: return
    concs = sorted(d.concurrency.unique()); w = 0.38
    fig, ax = plt.subplots(figsize=(6, 4))
    for i, s in enumerate(["bf16-nocache", "bf16"]):
        dd = d[d.server == s].set_index("concurrency").reindex(concs)
        xs = [j + (i - 0.5) * w for j in range(len(concs))]
        ax.bar(xs, dd.p50_ttft, w * 0.92, color=COLOR[s] if s != "bf16" else BLUE,
               label="prefix cache off" if s == "bf16-nocache" else "prefix cache on (default)")
        for x, v, h in zip(xs, dd.p50_ttft, dd.prefix_hit_rate):
            if not math.isnan(v):
                ax.annotate(f"{v:.0f} ms\nhit {h:.0%}" if h == h and h is not None else f"{v:.0f} ms", (x, v),
                            ha="center", va="bottom", fontsize=8, color=INK2)
    ax.margins(y=0.18)
    ax.set_xticks(range(len(concs)), [str(c) for c in concs])
    ax.set(title="6k-token shared prefix + 500-token question", xlabel="concurrent requests", ylabel="p50 TTFT (ms)")
    ax.legend(frameon=False); fig.tight_layout(); fig.savefig(out / "3_prefix_cache.png"); plt.close(fig)


def spec_table(df):
    rows = []
    for wl in ["spec-code", "spec-chat"]:
        base = df[(df.server == "bf16") & (df.workload == wl)].set_index("concurrency")
        for s in ["bf16-eagle3", "bf16-ngram"]:
            for _, r in df[(df.server == s) & (df.workload == wl)].iterrows():
                if r.concurrency in base.index:
                    b = base.loc[r.concurrency]
                    rows.append({"workload": wl, "method": LABEL[s], "server": s, "concurrency": r.concurrency,
                                 "accept_len": r.accept_len, "tpot_speedup": b.p50_tpot / r.p50_tpot,
                                 "throughput_ratio": r.out_tok_s / b.out_tok_s})
    return pd.DataFrame(rows)


def fig_spec(st, out):
    if st.empty: return
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, wl in zip(axes, ["spec-code", "spec-chat"]):
        for s in ["bf16-eagle3", "bf16-ngram"]:
            d = st[(st.server == s) & (st.workload == wl)].sort_values("concurrency")
            if d.empty: continue
            ax.plot(d.concurrency, d.tpot_speedup, "o-", color=COLOR[s], label=LABEL[s])
            for _, r in d.iterrows():
                if r.accept_len == r.accept_len and r.accept_len:
                    ax.annotate(f"τ={r.accept_len:.2f}", (r.concurrency, r.tpot_speedup), textcoords="offset points",
                                xytext=(5, 5), fontsize=8, color=INK2)
        ax.axhline(1, color=GRAY, ls="--", lw=1)
        ax.set_xscale("log", base=2)
        ax.set(title="code rewrites (predictable output)" if wl == "spec-code" else "open questions",
               xlabel="concurrent requests", ylabel="per-user speedup vs no speculation (p50 TPOT)")
    axes[0].legend(frameon=False)
    fig.suptitle("Speculative decoding: τ = mean tokens accepted per step; below 1.0 = slower", fontsize=10, color=INK2)
    fig.tight_layout(); fig.savefig(out / "4_speculative.png"); plt.close(fig)


def cost_table(df, price):
    rows = []
    for wl in ["chat", "agent"]:
        for s in ["bf16", "fp8", "int4"]:
            d = df[(df.server == s) & (df.workload == wl)]
            if d.empty: continue
            ok = d[(d.p95_ttft <= SLO[wl]["ttft"]) & (d.p95_tpot <= SLO[wl]["tpot"])]
            best = ok.loc[ok.out_tok_s.idxmax()] if not ok.empty else None
            peak = d.loc[d.out_tok_s.idxmax()]
            usd = lambda tps: price / (tps * 3600) * 1e6 if price and tps else None
            rows.append({"workload": wl, "precision": LABEL[s],
                         "best_conc_in_slo": int(best.concurrency) if best is not None else None,
                         "out_tok_s_in_slo": round(best.out_tok_s) if best is not None else None,
                         "usd_per_1M_out_in_slo": usd(best.out_tok_s) if best is not None else None,
                         "usd_per_1M_total_in_slo": usd(best.total_tok_s) if best is not None else None,
                         "peak_out_tok_s": round(peak.out_tok_s), "peak_conc": int(peak.concurrency),
                         "usd_per_1M_out_at_peak": usd(peak.out_tok_s)})
    return pd.DataFrame(rows)


def md(df, digits=2):
    """A markdown table without the tabulate dependency."""
    fmt = lambda v: "–" if v is None or (isinstance(v, float) and math.isnan(v)) else f"{v:.{digits}f}" if isinstance(v, float) else str(v)
    return "\n".join(["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)] +
                     ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)])


def findings(df, env, servers, ct, st, price):
    L = [f"# Findings draft ({env['name']}, vLLM {env['vllm']})", "",
         "Auto-generated from the numbers. Rewrite in your own words before publishing; check every claim against the charts.", ""]
    L += ["## KV cache capacity (memory left after weights)", "", "| server | weights GiB | KV cache tokens | startup s |",
          "|---|---|---|---|"]
    for n, s in servers.items():
        L.append(f"| {n} | {s.get('weights_gib')} | {s.get('kv_cache_tokens'):,} | {s.get('startup_s')} |"
                 if s.get("kv_cache_tokens") else f"| {n} | {s.get('weights_gib')} | ? | {s.get('startup_s')} |")
    L += ["", f"## Cost per 1M tokens at ${price}/GPU-hour (SLO: p95 TTFT ≤ 1 s chat / 3 s agent, p95 TPOT ≤ 50 ms)", "",
          md(ct, 3) if not ct.empty else "(no sweep data)", ""]
    if not st.empty:
        L += ["## Speculative decoding", "", md(st.drop(columns="server")), ""]
    p = df[df.workload == "prefix"]
    if not p.empty:
        L += ["## Prefix caching", ""]
        for c in sorted(p.concurrency.unique()):
            on = p[(p.server == "bf16") & (p.concurrency == c)]; off = p[(p.server == "bf16-nocache") & (p.concurrency == c)]
            if len(on) and len(off):
                L.append(f"- concurrency {c}: p50 TTFT {off.p50_ttft.iloc[0]:.0f} ms → {on.p50_ttft.iloc[0]:.0f} ms with the cache "
                         f"(hit rate {on.prefix_hit_rate.iloc[0]:.0%})")
        L.append("")
    pre = df[df.preemptions > 0]
    if not pre.empty:
        L += ["## Preemptions (KV cache ran out; requests were paused and recomputed)", "",
              *[f"- {r.id}: {r.preemptions:.0f}" for r in pre.itertuples()], ""]
    return "\n".join(L)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("results", type=Path)
    ap.add_argument("--price", type=float, help="$/GPU-hour override")
    a = ap.parse_args()
    df, env, servers = load(a.results)
    out = a.results / "figures"; out.mkdir(exist_ok=True)
    price = price_for(env["name"], a.price)
    df.to_csv(a.results / "results.csv", index=False)
    fig_frontier(df, out, env["name"]); fig_ttft(df, out); fig_prefix(df, out)
    st = spec_table(df); fig_spec(st, out)
    ct = cost_table(df, price)
    (a.results / "findings.md").write_text(findings(df, env, servers, ct, st, price))
    print(f"{len(df)} runs · {env['name']} · ${price}/h")
    print(ct.to_string(index=False))
    print(f"wrote {a.results}/results.csv, findings.md, figures/")


if __name__ == "__main__":
    main()

In [ ]:
!{PY} runner.py --out "{OUT}" --plan | tail -4

## 4. Preflight (~5 min)

One server, one tiny run. It catches a broken install, a model that won't load or a benchmark flag that changed,
before the long run starts. The first start is the slowest: vLLM compiles the model and captures CUDA graphs.

In [ ]:
!{PY} runner.py --out "{OUT}" --preflight
print(open(OUT / "preflight/logs/server-bf16.log").read()[-1500:])

## 5. The full run, in the background

It keeps going if this tab disconnects. To resume after a disconnect, run sections 1–3 again and then this cell:
finished runs are skipped. The next cell watches progress.

In [ ]:
progress = OUT / "progress.log"
start_line = len(progress.read_text().splitlines()) if progress.exists() else 0
run = subprocess.Popen(f"nohup '{PY}' runner.py --out '{OUT}' > runner.out 2>&1", shell=True, start_new_session=True)
time.sleep(5); print(open("runner.out").read())

In [ ]:
# Watch. Prints new progress lines every 30 s until the run finishes. Safe to stop and re-run.
seen = start_line
while True:
    lines = progress.read_text().splitlines() if progress.exists() else []
    for l in lines[seen:]: print(l)
    seen = len(lines)
    if any(l.split("  ", 1)[-1].startswith("finished") for l in lines[start_line:]): break
    if run.poll() is not None:
        print("runner stopped without finishing; its output:\n" + open("runner.out").read()[-3000:]); break
    time.sleep(30)

### While it runs: predict first, then check

Write down a guess for each, then compare with the charts. The why is in the inference notes
(`knowledge/llm-inference/` in career-copilot).

1. **Sweep.** From 1 to 128 concurrent requests, total tokens/s grows a lot but each user's tokens/s falls only a
   little at first. *Why:* decode is memory-bound. Each step reads all 16 GB of weights whatever the batch, so extra
   requests ride along almost free until compute runs out. Guess the concurrency where per-user speed drops below
   20 tokens/s.
2. **Precision.** FP8 should beat BF16 everywhere on an H100 or L4 (half the bytes to read, native FP8 maths). INT4 (AWQ)
   should win at concurrency 1 (a quarter of the bytes) but lose its lead at high concurrency, where it has to dequantize
   to 16-bit before every matmul and the GPU is compute-bound. Check the server table: fewer weight bytes also means
   more KV cache.
3. **Agent prompts.** An 8k prompt is 8× the prefill of a 1k one, so the time to first token rises roughly 8× at
   concurrency 1 and climbs faster with load, because new prompts wait behind other prefills. This is the cost
   structure of an agent loop.
4. **Prefix cache.** With 6,000 shared tokens out of 6,500, the cache skips ~90% of prefill, so the time to first token
   should drop several-fold. Throughput barely changes, because decode was never the problem here.
5. **Speculative decoding.** At concurrency 1, both methods should speed up code rewrites (predictable text) more than
   open questions. At high concurrency the speedup falls below 1.0: the GPU no longer has spare compute to check the
   guesses. τ (accepted tokens per step) tells you how good the guesses were.

## 6. First look (on the GPU machine)

In [ ]:
!{PY} analyze.py "{OUT}"
from IPython.display import Image, Markdown, display
for f in sorted((OUT / "figures").glob("*.png")): display(Image(str(f)))
display(Markdown((OUT / "findings.md").read_text()))

## 7. Save and stop paying

1. Results are in `OUT` (on Colab that's Google Drive already). The next cell also makes a zip to download.
2. **Stop the machine.** Colab: *Runtime → Disconnect and delete runtime* (an idle runtime still uses compute units).
   RunPod or JarvisLabs: stop **and delete/terminate** the instance; a stopped pod still bills for its disk.

In [ ]:
import shutil
z = shutil.make_archive(str(WORK / "serving-results"), "zip", OUT)
print(z)
if COLAB:
    from google.colab import files; files.download(z)

## 8. After the GPU (on the Mac, no GPU needed)

Unzip into `llm-systems-lab/serving/results/<gpu>/`, then
`~/.venvs/serving/bin/python serving/analyze.py serving/results/<gpu>`. The write-up comes from `findings.md`,
rewritten in your own words, plus the four charts. The resume bullet takes its numbers from the cost table and
the speedups.